In [9]:
import torch
import torch.nn as nn
import joblib
import numpy as np

In [10]:


# 1. Model İskeletinin Tanımlanması 
# (PyTorch ağırlıkları yüklerken bu iskelete oturtur)
class AyakkabiTahminModeli(nn.Module):
    def __init__(self):
        super(AyakkabiTahminModeli, self).__init__()
        self.ag_yapisi = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.ag_yapisi(x)

# 2. Dosyadan Ağırlık Yükleyen Tahmin Fonksiyonu
def ayakkabi_numarasi_tahmin_et(kilo, boy, yas, cinsiyet, model_yolu="ayakkabi_modeli.pth", scaler_yolu="veri_olceklendirici.pkl"):
    
    # Boş model iskeletini oluştur
    model = AyakkabiTahminModeli()
    
    # Ağırlıkları diskteki dosyadan modele yükle
    model.load_state_dict(torch.load(model_yolu, weights_only=True))
    model.eval() # Dropout/BatchNorm gibi katmanları tahmin moduna sabitler
    
    # Eğitimde kullanılan veri ölçeklendiricisini dosyadan yükle
    scaler = joblib.load(scaler_yolu)
    
    # Gelen ham verileri eğitimdeki formata dönüştür
    cinsiyet_kod = 1 if cinsiyet.upper() == 'E' else 0
    boy_metre = boy / 100 if boy > 3 else boy
    
    # Modeli besleyecek Numpy dizisini oluştur ve ölçeklendir
    yeni_veri = np.array([[kilo, boy_metre, yas, cinsiyet_kod]])
    olcekli_veri = scaler.transform(yeni_veri)
    
    # Ölçeklenmiş veriyi PyTorch Tensörüne çevir
    veri_tensor = torch.tensor(olcekli_veri, dtype=torch.float32)
    
    # Türev hesaplamalarını kapatarak (no_grad) ileri yayılım ile tahmin yap
    with torch.no_grad():
        tahmin = model(veri_tensor)
        
    return round(tahmin.item(), 1)

# --- Kullanım Örneği ---
# Bu script'i çalıştırdığınızda ilgili .pth ve .pkl dosyaları aynı klasördeyse başarıyla sonuç verir.
sonuc = ayakkabi_numarasi_tahmin_et(kilo=82, boy=1.78, yas=28, cinsiyet="E")
print(f"Sistemin Tahmin Ettiği Ayakkabı Numarası: {sonuc}")

Sistemin Tahmin Ettiği Ayakkabı Numarası: 42.3


In [11]:

def ayakkabi_numarasi_tahmin_et_jit(kilo, boy, yas, cinsiyet, model_yolu="bagimsiz_model.pt", scaler_yolu="veri_olceklendirici.pkl"):
    # Doğrudan derlenmiş modeli yükle (Sınıf iskeletine gerek yok!)
    model = torch.jit.load(model_yolu)
    
    scaler = joblib.load(scaler_yolu)
    
    cinsiyet_kod = 1 if cinsiyet.upper() == 'E' else 0
    boy_metre = boy / 100 if boy > 3 else boy
    
    yeni_veri = np.array([[kilo, boy_metre, yas, cinsiyet_kod]])
    olcekli_veri = scaler.transform(yeni_veri)
    
    veri_tensor = torch.tensor(olcekli_veri, dtype=torch.float32)
    
    with torch.no_grad():
        tahmin = model(veri_tensor)
        
    return round(tahmin.item(), 1)

In [14]:
sonuc2 = ayakkabi_numarasi_tahmin_et_jit(kilo=82, boy=1.78, yas=28, cinsiyet="E")
print(f"Sistemin Tahmin Ettiği Ayakkabı Numarası: {sonuc}")

Sistemin Tahmin Ettiği Ayakkabı Numarası: 42.3


/home/ogretmen/Belgeler/2026_2027/oyg2a1/.venv/lib/python3.14/site-packages/torch/jit/_serialization.py:170: FutureWarning: `torch.jit.load` is not supported in Python 3.14+ and may break. Please switch to `torch.export`.
  warnings.warn(


In [18]:
sonuc2 = ayakkabi_numarasi_tahmin_et_jit(kilo=55, boy=1.73, yas=14, cinsiyet="E")
print(f"Sistemin Tahmin Ettiği Ayakkabı Numarası: {sonuc2}")

Sistemin Tahmin Ettiği Ayakkabı Numarası: 43.8


/home/ogretmen/Belgeler/2026_2027/oyg2a1/.venv/lib/python3.14/site-packages/torch/jit/_serialization.py:170: FutureWarning: `torch.jit.load` is not supported in Python 3.14+ and may break. Please switch to `torch.export`.
  warnings.warn(
